<a href="https://colab.research.google.com/github/mikhail-mat/mit-ocw_hands-on-deep-learning/blob/main/Standalone_Embeddings_GloVe_From_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Intro

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from scipy.sparse import lil_matrix

In [2]:
keras.utils.set_random_seed(42)

In [3]:
train_url = "https://www.dropbox.com/scl/fi/ito6bnl2yaf1uw0uqibzf/lyric_genre_train.csv?rlkey=04dkn5un2djza8x0bdmfnlw3u&st=y47qh8i4&dl=1"
val_url = "https://www.dropbox.com/scl/fi/xmywjzqsaa8n5sn1bs0t9/lyric_genre_val.csv?rlkey=hggbeo0s1iaxjpa6z80429xl9&st=6i7d8eau&dl=1"
test_url = "https://www.dropbox.com/scl/fi/fnocl69w9ojs9s5zb0xvf/lyric_genre_test.csv?rlkey=z4hjopw7vaihoh948cbb5mvdp&st=xwond7dp&dl=1"

train_df = pd.read_csv(train_url,index_col=0)
val_df = pd.read_csv(val_url,index_col=0)
test_df = pd.read_csv(test_url,index_col=0)

In [4]:
print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(48991, 2)
(16331, 2)
(21774, 2)


In [5]:
train_df.head()

,Lyric,Genre
0,"Oh, girl. I can't get ready (Can't get ready f...",Pop
1,We met on a rainy evening in the summertime. D...,Pop
2,We carried you in our arms. On Independence Da...,Rock
3,I know he loved you. A long time ago. I ain't ...,Pop
4,Paralysis through analysis. Yellow moral uncle...,Rock


## Implementing GloVe from scratch

In [6]:
text_vectorization = keras.layers.TextVectorization(
    max_tokens=500,
    output_sequence_length=300,
    output_mode='int'
)

In [ ]:
# build the co-occurence matrix:
#   go through the corpus text using the predefined window size e.g. 2: how often
#   words appear together in the window 2 words before and after the current word

In [7]:
text_vectorization.adapt(train_df['Lyric'])
vocab_size = text_vectorization.vocabulary_size()

In [ ]:
co_occurrence = lil_matrix((vocab_size, vocab_size))

vectorized_songs = text_vectorization(train_df['Lyric']).numpy()

for song in vectorized_songs:
    for i in range(len(song)):
        current = song[i]
        if current == 0:
          continue

        for j in range(i-2, i+3):
            if 0 <= j < len(song) and j != i:
                neighbor = song[j]
                if neighbor == 0:
                  continue
                co_occurrence[current, neighbor] += 1

co_occurrence = co_occurrence.tocoo()

from scipy.sparse import save_npz
save_npz("co_occurrence_matrix.npz", co_occurrence)

from google.colab import files
files.download("co_occurrence_matrix.npz")

In [9]:
from scipy.sparse import load_npz
co_occurrence = load_npz("co_occurrence_matrix.npz")

In [11]:
words = co_occurrence.row
context = co_occurrence.col

counts = co_occurrence.data
log_counts = np.log(counts)

embedding_dim = 100 # embedding vectors will be 100 long

In [18]:
# define a model by the formula: log(X(i,j)) = b(i) + b(j) + w(i)*w(j)

# the input are two integers representing the position in the co-occurence matrix
word_input = keras.layers.Input(shape=(1,))
context_input = keras.layers.Input(shape=(1,))

# create 100-long embedding vectors for the word and the context word
word_embedding = keras.layers.Embedding(
    input_dim=vocab_size, output_dim=embedding_dim
)(word_input)
context_embedding = keras.layers.Embedding(
    input_dim=vocab_size, output_dim=embedding_dim
)(context_input)

# add biases (natural occurence of both word and context word)
b_word = keras.layers.Embedding(input_dim=vocab_size, output_dim=1)(word_input)
b_context = keras.layers.Embedding(
    input_dim=vocab_size, output_dim=1
)(context_input)

# compute the dot product of the embedding vectors and add biases
dot = keras.layers.Dot(axes=-1)([word_embedding, context_embedding])
output = keras.layers.Add()([dot, b_word, b_context])
output = keras.layers.Flatten()(output)

model = keras.Model([word_input, context_input], output)

model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_7       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_12        │ (None, 1, 100)    │     50,000 │ input_layer_6[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_13        │ (None, 1, 100)    │     50,000 │ input_layer_7[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot_3 (Dot)         │ (None, 1, 1)      │          0 │ embedding_12[0][… │
│                     │                   │            │ embedding_13[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_14        │ (None, 1, 1)      │        500 │ input_layer_6[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_15        │ (None, 1, 1)      │        500 │ input_layer_7[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_5 (Add)         │ (None, 1, 1)      │          0 │ dot_3[0][0],      │
│                     │                   │            │ embedding_14[0][… │
│                     │                   │            │ embedding_15[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 1)         │          0 │ add_5[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 101,000 (394.53 KB)

 Trainable params: 101,000 (394.53 KB)

 Non-trainable params: 0 (0.00 B)

In [19]:
model.compile(
    optimizer='adam',
    loss='mse'
)

In [20]:
model.fit(
    [words, context],
    log_counts,
    epochs=10,
    batch_size=512
)

Epoch 1/10
446/446 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 8.9030
Epoch 2/10
446/446 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.0878
Epoch 3/10
446/446 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.9667
Epoch 4/10
446/446 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.8829
Epoch 5/10
446/446 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.8339
Epoch 6/10
446/446 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.7629
Epoch 7/10
446/446 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6972
Epoch 8/10
446/446 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6479
Epoch 9/10
446/446 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6100
Epoch 10/10
446/446 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5777


In [21]:
word_embeddings = model.layers[2].get_weights()[0]
context_embeddings = model.layers[3].get_weights()[0]

final_embeddings = word_embeddings + context_embeddings

In [22]:
final_embeddings.shape

(500, 100)

In [ ]:
# classifier

input = keras.layers.Input()